In [4]:
import numpy as np
from PIL import Image


In [5]:
def replace_color(
    input_path: str,
    output_path: str,
    target_color: tuple[int, int, int] | tuple[int, int, int, int],
    replacement_color: tuple[int, int, int, int],
    tolerance: int = 0,
) -> None:
    """Replaces all instances of target_color with replacement_color in an image.

    :param input_path: Path to the source image.
    :param output_path: Destination path for the modified image (.png recommended).
    :param target_color: (R, G, B) or (R, G, B, A) color to replace (0-255).
    :param replacement_color: (R, G, B, A) replacement color (0-255).
    :param tolerance: Threshold variation per RGB channel (0 for exact match).
    """
    # Open image and ensure RGBA mode so transparency is supported
    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    # Extract color channels
    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_color[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        # Match colors within a tolerance window (useful for anti-aliased/JPEG edges)
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # If target specifies an alpha channel, factor that into the match
    if len(target_color) == 4:
        match_mask = match_mask & (data[..., 3] == target_color[3])

    # Assign the new RGBA color to matching pixels
    data[match_mask] = replacement_color

    # Save output
    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")

In [6]:
def hex_to_rgba(hex_code: str, default_alpha: int = 255) -> tuple[int, int, int, int]:
    """Converts 6-digit or 8-digit hex strings into an RGBA tuple."""
    hex_code = hex_code.lstrip("#")
    if len(hex_code) == 6:
        r, g, b = (int(hex_code[i : i + 2], 16) for i in (0, 2, 4))
        return (r, g, b, default_alpha)
    elif len(hex_code) == 8:
        return tuple(int(hex_code[i : i + 2], 16) for i in (0, 2, 4, 6))
    raise ValueError(f"Invalid hex color format: {hex_code}")


def replace_hex_color(
    input_path: str,
    output_path: str,
    target_hex: str = "#000000",
    replacement_hex: str = "#1A76F9",
    tolerance: int = 0,
) -> None:
    """Replaces occurrences of target_hex with replacement_hex in an image and saves as PNG."""
    target_rgba = hex_to_rgba(target_hex)
    replacement_rgba = hex_to_rgba(replacement_hex)

    img = Image.open(input_path).convert("RGBA")
    data = np.array(img)

    r, g, b = data[..., 0], data[..., 1], data[..., 2]
    tr, tg, tb = target_rgba[:3]

    if tolerance == 0:
        match_mask = (r == tr) & (g == tg) & (b == tb)
    else:
        match_mask = (
            (np.abs(r.astype(int) - tr) <= tolerance)
            & (np.abs(g.astype(int) - tg) <= tolerance)
            & (np.abs(b.astype(int) - tb) <= tolerance)
        )

    # Respect source alpha if the target specifies an explicit alpha value
    if len(target_hex.lstrip("#")) == 8:
        match_mask = match_mask & (data[..., 3] == target_rgba[3])

    data[match_mask] = replacement_rgba

    output_img = Image.fromarray(data, mode="RGBA")
    output_img.save(output_path, format="PNG")

In [ ]:
def tint_image(
    input_path: str,
    output_path: str,
    tint_hex: str = "#1A76F9",
    mode: str = "duotone",
) -> None:
    """Recolors all non-transparent pixels to match the given tint while preserving tone/shading.

    :param input_path: Path to source image.
    :param output_path: Destination path for PNG output.
    :param tint_hex: Primary target hex color.
    :param mode: 'duotone' maps shadows -> tint_color -> highlights (preserves full contrast).
                 'multiply' scales luminance directly by tint_color (best for icons/graphics on dark bases).
    """
    tint_rgba = hex_to_rgba(tint_hex)
    tint_rgb = np.array(tint_rgba[:3], dtype=np.float32)

    img = Image.open(input_path).convert("RGBA")
    data = np.array(img, dtype=np.float32)

    rgb = data[..., :3]
    alpha = data[..., 3]

    # Standard perceived luminance (ITU-R BT.601) normalized between 0.0 and 1.0
    lum = (0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]) / 255.0

    if mode == "duotone":
        # Two-segment linear interpolation:
        # Shadows (0.0 to 0.5) scale from Black (0, 0, 0) up to the Target Tint
        # Highlights (0.5 to 1.0) scale from the Target Tint up to Pure White (255, 255, 255)
        shadow_mask = lum < 0.5
        highlight_mask = ~shadow_mask

        new_rgb = np.zeros_like(rgb)

        # Shadows: 0.0 -> black, 0.5 -> tint_rgb
        factor_shadow = (lum[shadow_mask, None] * 2.0)
        new_rgb[shadow_mask] = factor_shadow * tint_rgb

        # Highlights: 0.5 -> tint_rgb, 1.0 -> white
        factor_highlight = ((lum[highlight_mask, None] - 0.5) * 2.0)
        new_rgb[highlight_mask] = tint_rgb + factor_highlight * (255.0 - tint_rgb)

    elif mode == "multiply":
        # Multiplies normalized luminance directly by the target tint (max output brightness = tint_hex)
        new_rgb = lum[..., None] * tint_rgb

    else:
        raise ValueError("Mode must be either 'duotone' or 'multiply'")

    # Mask out fully transparent pixels to prevent edge spill
    visible_mask = alpha > 0
    data[visible_mask, :3] = np.clip(new_rgb[visible_mask], 0, 255)

    # Convert back to uint8 RGBA and save
    output_img = Image.fromarray(data.astype(np.uint8), mode="RGBA")
    output_img.save(output_path, format="PNG")

In [ ]:
input_locations = ["/home/david/Git/SDLT_Number_Crunching/Images/scales.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/checklist.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/book.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/spyglass.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/clusters.drawio.png"]

intermediate_save_locations = ["/home/david/Git/SDLT_Number_Crunching/Images/scales_intermediate.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/checklist_intermediate.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/book_intermediate.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/spyglass_intermediate.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/clusters_intermediate.drawio.png"]

output_locations = ["/home/david/Git/SDLT_Number_Crunching/Images/scales_cleaned.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/checklist_cleaned.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/book_cleaned.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/spyglass_cleaned.drawio.png", "/home/david/Git/SDLT_Number_Crunching/Images/clusters_cleaned.drawio.png"]

WHITE = (255, 255, 255)
TRANSPARENT = (0, 0, 0, 0)

for count, location in enumerate(input_locations):

    # Set tolerance=0 for exact white, or e.g. tolerance=10-20 to clean up near-white compression artifacts
    replace_color(
        input_path=location,
        output_path=intermediate_save_locations[count],
        target_color=WHITE,
        replacement_color=TRANSPARENT,
        tolerance=100,
    )

    tint_image(
        input_path=intermediate_save_locations[count],
        output_path=output_locations[count],
        tint_hex="#1A76F9",
        mode="duotone",  # Use 'duotone' for realistic photos/gradients, 'multiply' for flat line art
    )


    # replace_hex_color(
    #     input_path=location,
    #     output_path=location,
    #     target_hex="#000000",
    #     replacement_hex="#1A76F9",
    #     tolerance=15,  # Catches dark anti-aliased edges often found around black text/shapes
    # )